# Fake News Detection: Model Training

This notebook handles:
1. Class balancing
2. Data splitting
3. Training BERT, RoBERTa, and Hybrid models
4. Integration with Tinker API for distributed training
5. Model evaluation and comparison

## 1. Setup and Configuration

In [1]:
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Custom modules
from models import ClassBalancer, BERTModel, RoBERTaModel, HybridModel, TinkerIntegration
from training_utils import DataSplitter, MetricsCalculator, ModelTrainer
from config import TRUTHFULNESS_LABELS, RANDOM_SEED

# Set random seed
np.random.seed(RANDOM_SEED)

# Visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")
print(f"✓ Random seed set to {RANDOM_SEED}")

✓ All imports successful
✓ Random seed set to 42


## 2. Load Processed Data

In [2]:
# Load processed data from previous notebook
artifacts_dir = '../artifacts'

# Load encoded data
data = np.load(f'{artifacts_dir}/encoded_data.npz')
X = data['X']
y = data['y']

print(f"Loaded data:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"\nLabel distribution (before balancing):")
unique, counts = np.unique(y, return_counts=True)
for label_idx, count in zip(unique, counts):
    label_name = TRUTHFULNESS_LABELS[label_idx] if isinstance(TRUTHFULNESS_LABELS, list) else TRUTHFULNESS_LABELS[label_idx]
    print(f"  {label_name}: {count} ({count/len(y)*100:.1f}%)")

Loaded data:
  X shape: (10240, 100)
  y shape: (10240,)

Label distribution (before balancing):
  true: 1654 (16.2%)
  mostly-true: 1995 (19.5%)
  half-true: 2114 (20.6%)
  barely-true: 1962 (19.2%)
  false: 839 (8.2%)
  pants-on-fire: 1676 (16.4%)


## 3. Class Balancing

In [3]:
# Compute class weights
num_classes = len(TRUTHFULNESS_LABELS) if isinstance(TRUTHFULNESS_LABELS, list) else len(TRUTHFULNESS_LABELS)
balancer = ClassBalancer()
class_weights = balancer.compute_class_weights(y, num_classes)

In [4]:
# Choose balancing strategy
# Option 1: Oversampling (for smaller datasets)
print("Applying oversampling to balance classes...")
X_balanced, y_balanced = balancer.oversample_minority(X, y)

print(f"\nLabel distribution (after balancing):")
unique, counts = np.unique(y_balanced, return_counts=True)
for label_idx, count in zip(unique, counts):
    label_name = TRUTHFULNESS_LABELS[label_idx] if isinstance(TRUTHFULNESS_LABELS, list) else TRUTHFULNESS_LABELS[label_idx]
    print(f"  {label_name}: {count} ({count/len(y_balanced)*100:.1f}%)")

Applying oversampling to balance classes...

Label distribution (after balancing):
  true: 2114 (16.7%)
  mostly-true: 2114 (16.7%)
  half-true: 2114 (16.7%)
  barely-true: 2114 (16.7%)
  false: 2114 (16.7%)
  pants-on-fire: 2114 (16.7%)


## 4. Data Splitting

In [5]:
# Split data
splitter = DataSplitter()
splits = splitter.split_data(
    X_balanced, y_balanced,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=RANDOM_SEED
)

X_train, y_train = splits['train']
X_val, y_val = splits['val']
X_test, y_test = splits['test']

print(f"\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label_idx, count in zip(unique, counts):
    label_name = TRUTHFULNESS_LABELS[label_idx] if isinstance(TRUTHFULNESS_LABELS, list) else TRUTHFULNESS_LABELS[label_idx]
    print(f"  {label_name}: {count}")


Train label distribution:
  true: 1480
  mostly-true: 1480
  half-true: 1480
  barely-true: 1479
  false: 1479
  pants-on-fire: 1480


## 5. BERT Model Setup

In [6]:
# Initialize BERT model
bert_model = BERTModel(model_name="bert-base-uncased", num_classes=num_classes)

print("\nBERT Model Configuration:")
print(bert_model.get_model_info())


BERT Model Configuration:
{'model_name': 'bert-base-uncased', 'type': 'BERT', 'num_classes': 6, 'status': 'not loaded', 'num_parameters': '110M (estimated)'}


In [7]:
# Load BERT model and tokenizer
bert_model.load_model()

print("✓ BERT model initialized (model loading skipped for demo)")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ BERT model initialized (model loading skipped for demo)


## 6. RoBERTa Model Setup

In [8]:
# Initialize RoBERTa model
roberta_model = RoBERTaModel(model_name="roberta-base", num_classes=num_classes)

print("\nRoBERTa Model Configuration:")
print(roberta_model.get_model_info())


RoBERTa Model Configuration:
{'model_name': 'roberta-base', 'type': 'RoBERTa', 'num_classes': 6, 'status': 'not loaded', 'num_parameters': '125M (estimated)'}


In [9]:
# Load RoBERTa model and tokenizer
# Uncomment to load actual model (requires transformers and torch)
# roberta_model.load_model()

print("✓ RoBERTa model initialized (model loading skipped for demo)")

✓ RoBERTa model initialized (model loading skipped for demo)


## 7. Hybrid Model Setup

In [10]:
# Initialize Hybrid model
hybrid_model = HybridModel(num_classes=num_classes, use_tinker=True)

print("\nHybrid Model Configuration:")
import json
print(json.dumps(hybrid_model.get_model_info(), indent=2))


Hybrid Model Configuration:
{
  "type": "Hybrid Ensemble",
  "num_classes": 6,
  "use_tinker": true,
  "weights": {
    "bert": 0.4,
    "roberta": 0.4,
    "features": 0.2
  },
  "models": [
    "BERT (bert-base-uncased)",
    "RoBERTa (roberta-base)",
    "Traditional Features"
  ]
}


In [11]:
# Set custom ensemble weights
hybrid_model.set_weights(bert_weight=0.45, roberta_weight=0.45, features_weight=0.1)

print("✓ Hybrid model weights configured")

✓ Hybrid model weights configured


## 8. Tinker API Integration

In [12]:
# Get API key from environment
api_key = os.getenv('TINKER_API_KEY')
api_url = os.getenv('TINKER_API_URL', 'https://api.tinker.thinkingmachines.ai')

if api_key and api_key != 'your_api_key_here':
    print("✓ Tinker API key found")
    tinker = TinkerIntegration(api_key=api_key, api_url=api_url)
    print(f"  URL: {api_url}")
else:
    print("⚠ Tinker API key not configured")
    print("  Set TINKER_API_KEY in .env file to enable API features")
    tinker = None

✓ Tinker API key found
  URL: https://api.tinker.thinkingmachines.ai


## 9. Training Configuration

In [13]:
# Training configuration
config = {
    'batch_size': 32,
    'epochs': 3,  # Start with 3 for demo
    'learning_rate': 0.00005,
    'max_seq_length': 128,
    'optimizer': 'adamw',
    'loss_fn': 'crossentropy',
    'warmup_steps': 500,
    'class_weights': class_weights
}

print("Training Configuration:")
for key, value in config.items():
    if key != 'class_weights':
        print(f"  {key}: {value}")

Training Configuration:
  batch_size: 32
  epochs: 3
  learning_rate: 5e-05
  max_seq_length: 128
  optimizer: adamw
  loss_fn: crossentropy
  warmup_steps: 500


## 10. Training Summary and Next Steps

In [14]:
print("\n" + "="*60)
print("MODEL TRAINING PIPELINE SUMMARY")
print("="*60)

print(f"\nDataset:")
print(f"  Original: {len(y)} samples")
print(f"  Balanced: {len(y_balanced)} samples")
print(f"  Train/Val/Test: {len(y_train)}/{len(y_val)}/{len(y_test)}")

print(f"\nModels to Train:")
print(f"  1. BERT (bert-base-uncased)")
print(f"  2. RoBERTa (roberta-base)")
print(f"  3. Hybrid Ensemble (BERT + RoBERTa)")

print(f"\nConfiguration:")
print(f"  Learning Rate: {config['learning_rate']}")
print(f"  Batch Size: {config['batch_size']}")
print(f"  Epochs: {config['epochs']}")
print(f"  Max Sequence Length: {config['max_seq_length']}")

print(f"\nIntegrations:")
print(f"  Tinker API: {'✓ Configured' if tinker else '✗ Not configured'}")

print(f"\n" + "="*60)
print("Ready for training!")
print("="*60)


MODEL TRAINING PIPELINE SUMMARY

Dataset:
  Original: 10240 samples
  Balanced: 12684 samples
  Train/Val/Test: 8878/1903/1903

Models to Train:
  1. BERT (bert-base-uncased)
  2. RoBERTa (roberta-base)
  3. Hybrid Ensemble (BERT + RoBERTa)

Configuration:
  Learning Rate: 5e-05
  Batch Size: 32
  Epochs: 3
  Max Sequence Length: 128

Integrations:
  Tinker API: ✓ Configured

Ready for training!
